In [2]:
import numpy as np
import rasterio
from rasterio.features import sieve
from scipy.ndimage import median_filter
from pathlib import Path

In [ ]:
result = np.zeros_like(stack)
result[0] = stack[0]
result[1:] = stack[1:] & stack[:-1] 
#result = np.logical_and.accumulate(stack, axis=0)

In [ ]:
import numpy as np
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape


def raster_to_polygons(binary_raster, transform, crs):
    """Convert pixels == 1 of a binary raster into polygons."""
    binary_raster = binary_raster.astype(np.uint8)
    mask = binary_raster == 1

    geoms = [
        shape(geom)
        for geom, value in shapes(binary_raster, mask=mask, transform=transform)
    ]
    return gpd.GeoDataFrame({"value": [1] * len(geoms)}, geometry=geoms, crs=crs)


version = "v0.1"

for year, layer in zip(valid_years, result):
    gdf = raster_to_polygons(layer, profile["transform"], profile["crs"])

    if gdf.empty:
        print(f"{year}: no glacier pixels left, skipping.")
        continue

    out_file = files[year].with_name(
        f"s1_s2_glacier_outlines_{year}_{version}_tsmooth.shp"
    )
    gdf.to_file(out_file)

    print(f"{year}: {len(gdf)} polygons, {gdf.area.sum() / 1e6:8.2f} km² -> {out_file.name}")

In [ ]:
import glob

def find_files(input_folder, pattern):
    #Find files in the input folder matching the given pattern
    return sorted(glob.glob(str(input_folder / pattern)))

In [3]:
BASE = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/sentinel_data/S1_coherence")
YEARS= [2016, 2017, 2018, 2019, 2020, 2021, 2022,2023, 2024, 2025, 2026]

In [4]:
import rasterio
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

def create_histogram(year, offset=0.07, bins=256, smooth=5):
    output_folder = BASE / str(year)
    files = list(output_folder.glob(f"coh_mosaic_*{year}.tiff"))
    lines = []

    for f in files:
        with rasterio.open(f) as src:
            data = src.read(1, masked=True).compressed()
        data = data[np.isfinite(data) & (data > 0) & (data <= 1)]

        counts, edges = np.histogram(data, bins=bins, range=(0, 1))
        counts = uniform_filter1d(counts.astype(float), size=smooth)  # suppress noise
        centers = (edges[:-1] + edges[1:]) / 2

        peaks, _ = find_peaks(counts, prominence=0.05 * counts.max())
        first_peak = round(float(centers[peaks[0]]), 2)
        threshold = round(first_peak + offset, 2)

        line = f"{year} | {f.name} | first peak = {first_peak:.2f} | threshold = {threshold:.2f}"
        print(line)
        lines.append(line)

    return lines

all_lines = []
for year in YEARS:
    all_lines += create_histogram(year)




2016 | coh_mosaic_ascending_2016.tiff | first peak = 0.33 | threshold = 0.40
2016 | coh_mosaic_descending_2016.tiff | first peak = 0.33 | threshold = 0.40
2017 | coh_mosaic_ascending_2017.tiff | first peak = 0.37 | threshold = 0.44
2017 | coh_mosaic_descending_2017.tiff | first peak = 0.36 | threshold = 0.43
2018 | coh_mosaic_descending_2018.tiff | first peak = 0.42 | threshold = 0.49
2018 | coh_mosaic_ascending_2018.tiff | first peak = 0.41 | threshold = 0.48
2019 | coh_mosaic_ascending_2019.tiff | first peak = 0.36 | threshold = 0.43
2019 | coh_mosaic_descending_2019.tiff | first peak = 0.42 | threshold = 0.49
2020 | coh_mosaic_descending_2020.tiff | first peak = 0.40 | threshold = 0.47
2020 | coh_mosaic_ascending_2020.tiff | first peak = 0.36 | threshold = 0.43
2021 | coh_mosaic_descending_2021.tiff | first peak = 0.38 | threshold = 0.45
2021 | coh_mosaic_ascending_2021.tiff | first peak = 0.38 | threshold = 0.45
2022 | coh_mosaic_descending_2022.tiff | first peak = 0.26 | threshold

In [15]:
out_file = BASE / "coherence_thresholds.txt"
out_file.write_text("\n".join(all_lines) + "\n")
print(f"\nSaved to {out_file}")


Saved to /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/coherence/coherence_thresholds.txt
